# 🧠 ParagAI: World-Class LLM Training on Google Colab (GPU / TPU)

Welcome to **ParagAI** — a first-principles, industrial-grade Large Language Model built from scratch in Python and PyTorch. ParagAI matches Andrej Karpathy's `nanochat` speed and performance, featuring:

- ⚡ **Hardware Acceleration:** `torch.compile` kernel fusion + native `bfloat16` mixed precision + gradient accumulation.
- 🌐 **Multilingual Indic BPE Tokenizer:** Native subword coverage for English, Bengali, Hindi, Tamil, and Telugu with guaranteed 256-byte UTF-8 fallback.
- 📊 **Live Telemetry:** Real-time tokens/sec, VRAM tracking, gradient norms, and training loss dashboard.
- 🧪 **Benchmark Suite:** Automated evaluation across Perplexity, Multilingual, Tool Use, and Story Generation.
- 🚀 **Dual Deployment:** Export to **Ollama** (GGUF with GPT-2 Unicode byte mapping) and **vLLM** (Hugging Face SafeTensors).

---

### ⚙️ Step 0: Set Up Free GPU Runtime
1. In the top menu, navigate to **Runtime** > **Change runtime type**.
2. Under *Hardware accelerator*, select **T4 GPU** (or **A100 / V100** if you have Colab Pro).
3. Click **Save**.

## 📥 1. Clone ParagAI Repository

In [ ]:
import os

if not os.path.exists('/content/ParagAI'):
    !git clone https://github.com/paragmm/ParagAI.git
%cd /content/ParagAI
!git checkout feature/technology-upgrade || git checkout feature/paragAI-pro || true
!git pull

## 📦 2. Install Dependencies in Editable Mode

In [ ]:
!pip install --upgrade pip
!pip install -e ".[dev]"
!pip install rich safetensors gguf datasets

## 🩺 3. Run Pre-Flight System Diagnostics (`doctor.py`)
Verify GPU availability, VRAM capacity, PyTorch version, `torch.compile` compatibility, and tokenizer integrity.

In [ ]:
!python scripts/doctor.py

## 📚 4. Prepare Multilingual & TinyStories SFT Dataset
Download high-quality story data from Hugging Face `roneneldan/TinyStories` and format with ChatML and replay mixing.

In [ ]:
# 4.1 Fetch 3,000 TinyStories examples
!python scripts/fetch_tinystories.py --num_stories 3000 --output data/raw/stories/tinystories_colab.txt

# 4.2 Synthesize ChatML conversations with Experience Replay mixing
!python scripts/prepare_tinystories_sft.py \
    --input data/raw/stories/tinystories_colab.txt \
    --output data/processed/stories_sft.txt \
    --replay_mix True \
    --replay_ratio 0.20

# 4.3 Merge all multi-task data into combined corpus
!python scripts/prepare_data.py --no_tags

## 🔤 5. Train High-Speed Multilingual Indic + English BPE Tokenizer
Trains the `IndicBPETokenizer` across English, Bengali, and Hindi with guaranteed 256-byte fallback.

In [ ]:
!python scripts/train_multilingual_bpe.py \
    --input "data/raw/**/*.txt" \
    --vocab_size 4096 \
    --output checkpoints/tokenizer_indic.json \
    --sample_mb 5

## 🚀 6. Accelerated GPU Training with Live Telemetry
Trains ParagAI using `bfloat16` mixed precision, gradient accumulation, and real-time ASCII telemetry dashboard.

In [ ]:
# Run training on Colab GPU with train_fast config
!python scripts/train.py \
    --data data/processed/combined_corpus.txt \
    --config configs/train_fast.yaml \
    --model_config configs/model_small.yaml \
    --tokenizer_type bpe \
    --max_iters 1000

## 💬 7. Interactive Chat & Multi-Task Generation
Test your freshly trained model interactively inside the notebook using the ChatML protocol.

In [ ]:
import json
import torch
from paragai.config import ModelConfig
from paragai.inference.generate import Generator
from paragai.model.transformer import ParagAIModel
from paragai.tokenizer.bpe_tokenizer import BPETokenizer
from paragai.tokenizer.char_tokenizer import CharTokenizer
from paragai.tokenizer.indic_bpe import IndicBPETokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('checkpoints/best_model.pt', map_location=device)
model_config = ModelConfig(**checkpoint['model_config'])
model = ParagAIModel(model_config)
state_dict = {k: v for k, v in checkpoint['model_state_dict'].items() if not k.endswith('.attn.mask')}
model.load_state_dict(state_dict, strict=False)

# Auto-detect tokenizer type
tok_path = 'checkpoints/tokenizer.json' if os.path.exists('checkpoints/tokenizer.json') else 'checkpoints/tokenizer_indic.json'
with open(tok_path, 'r', encoding='utf-8') as f:
    tok_data = json.load(f)
if tok_data.get('tokenizer_class') == 'indic_bpe':
    tokenizer = IndicBPETokenizer.load(tok_path)
elif tok_data.get('type') == 'bpe':
    tokenizer = BPETokenizer.load(tok_path)
else:
    tokenizer = CharTokenizer.load(tok_path)

generator = Generator(model, tokenizer, device=device)

# Test sample generation
prompts = [
    "Hello! Who are you?",
    "Write a short creative story about a brave astronaut.",
    "আপনি কেমন আছেন? একটি ছোট গল্প বলুন।",
]

for p in prompts:
    formatted = f"<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n"
    response = generator.generate(formatted, max_new_tokens=100, temperature=0.3, include_prompt=False)
    clean = response.split('<|im_end|>')[0].strip()
    print(f"\n[Prompt]    {p}")
    print(f"[Response]  {clean}")

## 📊 8. Run Benchmark Evaluation Suite (`eval.py`)
Measures Perplexity, Multilingual Proficiency, Tool Adherence, and Narrative Story Openings.

In [ ]:
!python scripts/eval.py \
    --checkpoint checkpoints/best_model.pt \
    --tokenizer checkpoints/tokenizer.json \
    --tasks multiling,tools,story

## 📦 9. Universal Export: SafeTensors (vLLM) & GGUF (Ollama)

In [ ]:
# 9.1 Export to Hugging Face SafeTensors with self-verification
!python scripts/export_hf.py \
    --checkpoint checkpoints/best_model.pt \
    --output_dir checkpoints/export_hf \
    --verify

# 9.2 Export to GGUF with standard GPT-2 Unicode byte mapping (Zero UNK_BYTE!)
!python scripts/export_gguf.py \
    --input_dir checkpoints/export_hf \
    --tokenizer checkpoints/tokenizer.json \
    --output checkpoints/paragai.gguf

## 💾 10. Download Checkpoint & GGUF to Your Local Computer
Download your trained `.pt` and `.gguf` files directly to your computer so you can run `ollama run paragai:latest` locally offline!

In [ ]:
from google.colab import files

print("Downloading checkpoints/best_model.pt...")
files.download("checkpoints/best_model.pt")

print("Downloading checkpoints/tokenizer.json...")
files.download("checkpoints/tokenizer.json")

if os.path.exists("checkpoints/paragai.gguf"):
    print("Downloading checkpoints/paragai.gguf for Ollama...")
    files.download("checkpoints/paragai.gguf")

if os.path.exists("checkpoints/eval_results.json"):
    files.download("checkpoints/eval_results.json")